# Everpure | Install base monetization — Phase 1
**Observed-data foundation, as of 2026-10-09.** This notebook covers source hygiene, hierarchy, synthetic-date diagnostics, descriptive SQL, and a separate synthetic revenue-decay demonstration. The business brief defines dormancy using two fiscal years; the fiscal boundary is unavailable. Historical spend is a recorded source field, not recoverable revenue.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from everpure_case.config import SOURCE_COLUMNS
from everpure_case.ingestion import read_source
from everpure_case.pipeline import run_pipeline
from everpure_case.synthetic_demo import LABEL, run_synthetic_demo
OUT = ROOT / 'outputs' / 'phase1'
metrics = run_pipeline(ROOT)
raw = read_source(ROOT / 'data/raw/Case_Study_1_Install_Base_Reactivation.csv')
display(Markdown(f"**Source:** {metrics['source_rows']:,} rows; {metrics['unique_account_ids']:,} unique IDs; SHA-256 `{metrics['source_sha256']}`."))

## 1. Source contract and cleaning
The source fields remain intact in the cleaned output. Typed and normalized columns are appended.

In [ ]:
display(pd.DataFrame({'source_column': SOURCE_COLUMNS, 'nonempty_rows': [raw[c].ne('').sum() for c in SOURCE_COLUMNS], 'distinct_raw_values': [raw[c].nunique() for c in SOURCE_COLUMNS]}))
clean = pd.read_csv(OUT / 'cleaned_accounts.csv')
display(clean[['account_id', 'region', 'region_clean', 'sfdc_health_score', 'health_clean', 'partner_managed', 'partner_managed_clean', 'open_support_tickets', 'tickets_clean']].head())

In [ ]:
exceptions = pd.read_csv(OUT / 'data_quality_exceptions.csv')
display(exceptions.groupby(['issue_code', 'resolution']).size().rename('exception_rows').reset_index().sort_values('exception_rows', ascending=False))
account_quality = pd.read_csv(OUT / 'account_quality_summary.csv')
display(account_quality[['account_id', 'exception_count', 'manual_review_count', 'left_unknown_count', 'auto_resolved_count']].sort_values('exception_count', ascending=False).head())
display(Markdown(f"**Unique accounts with at least one exception:** {metrics['accounts_with_quality_exceptions']:,}."))

## 2. Inactivity and date forensics
Calendar eligibility means last booking on or before 2024-10-09. It is a proxy, not confirmed two-fiscal-year eligibility. The source-reported years are a separate definition; an unknown comparison is not an agreement. In every source row, booking date equals 2019-01-01 plus (numeric account ID - 1000) days, and contract start is 365 days earlier. Date-window outputs therefore collapse to ID arithmetic and are diagnostics only.

In [ ]:
recon = pd.read_csv(OUT / 'inactivity_reconciliation.csv')
display(recon)
sensitivity = pd.read_csv(OUT / 'analysis_date_sensitivity.csv')
display(sensitivity)
display(clean['eligibility_comparison_status'].value_counts())
display(Markdown(f"Booking dates: **{metrics['booking_min']} to {metrics['booking_max']}**. Daily sequence: **{metrics['booking_sequential_daily']}**. Contract to booking gap: **{metrics['contract_to_booking_days_unique']} days** for every row. Exact date/ID sequence: **{metrics['booking_date_exact_account_id_sequence']}**. Definition conflicts: **{metrics['eligibility_disagreement_rows']:,}**."))

## 3. Hierarchy and additive spend proxy
The resolver handles multiple levels within Python's recursion limit. Group totals sum each source row once; they are **not verified consolidated revenue**. For 291 of 298 referenced parents, parent-row spend is below direct-child spend sum. This argues against simple full-child inclusion, without establishing the true accounting rule.

In [ ]:
hierarchy = pd.read_csv(OUT / 'account_hierarchy.csv')
parents = pd.read_csv(OUT / 'parent_spend_summary.csv')
parent_comparison = pd.read_csv(OUT / 'parent_child_spend_comparison.csv')
display(hierarchy.hierarchy_status.value_counts().rename('account_rows').to_frame())
display(parents[['root_account_id', 'account_count', 'direct_child_count', 'additive_recorded_spend_usd', 'mean_recorded_spend_per_account_usd', 'region_count', 'partner_ownership_conflict']].head(10))

## 4. DuckDB source diagnostics
SQL files execute against cleaned accounts and a one-to-one hierarchy mapping. `LAG` and `LEAD` compare adjacent *accounts' single last-booking dates* within a group. Here their date gaps equal account-ID differences, so they are not behavioral frequency or decay.

In [ ]:
display(Markdown('### Booking-window SQL\n```sql\n' + (ROOT / 'sql/window_analytics.sql').read_text(encoding='utf-8') + '\n```'))
spend = pd.read_csv(OUT / 'spend_concentration.csv')
region = pd.read_csv(OUT / 'regional_spend.csv')
opportunity = pd.read_csv(OUT / 'account_opportunity.csv')
window = pd.read_csv(OUT / 'window_analytics.csv')
display(region[['region', 'account_count', 'recorded_spend_usd', 'spend_share']])
display(opportunity[['account_id', 'root_account_id', 'recorded_spend_usd', 'region_recorded_spend_rank', 'ownership_unresolved', 'health_unknown', 'eligibility_comparison_status']].head(10))
display(window[['root_account_id', 'account_id', 'booking_date', 'previous_account_booking_date', 'days_since_previous_account_booking', 'recency_rank_within_root']].head(10))

## 5. Controlled longitudinal SQL demonstration
**SYNTHETIC METHODOLOGY DEMONSTRATION — NOT OBSERVED EVERPURE REVENUE.** The separate fixture has four invented fiscal periods, a missing period, an observed zero, and additive parent-child revenue by fixture assumption. Consecutive percentage change is suppressed over a period gap or zero baseline; signed USD change per elapsed fiscal period is a separately labeled measure. No synthetic value enters the real CSV outputs.

In [ ]:
synthetic_account, synthetic_root = run_synthetic_demo(ROOT)
display(Markdown('**' + LABEL + '**'))
display(synthetic_account.loc[(synthetic_account.account_id.isin(['SYN_A', 'SYN_B', 'SYN_C'])) & (synthetic_account.fiscal_period_index.isin([2, 4])), ['account_id', 'fiscal_period_index', 'period_revenue_usd', 'previous_revenue_usd', 'elapsed_periods', 'absolute_change_usd', 'consecutive_pct_change', 'signed_change_usd_per_period', 'comparison_status']])
display(synthetic_root.loc[synthetic_root.root_account_id.eq('SYN_R1'), ['fiscal_period_index', 'observed_account_count', 'expected_account_count', 'observed_partial_revenue_usd', 'root_revenue_usd', 'consecutive_pct_change', 'comparison_status']])

## 6. Core descriptive figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
axes[0].barh(region['region'][::-1], region['recorded_spend_usd'][::-1] / 1e6, color='#287a78')
axes[0].set_xlabel('Recorded historical spend (USD millions)')
axes[0].set_title('Account-native region')
axes[1].bar(recon['source_reported_2y'], recon['account_rows'], color='#d69854')
axes[1].set_ylabel('Account rows')
axes[1].set_title('Source-reported 2-year status')
fig.suptitle('Descriptive source measures | 2026-10-09')
fig.tight_layout()
plt.show()

In [ ]:
total = metrics['raw_recorded_spend_usd']
top10 = spend.head(10)['additive_recorded_spend_usd'].sum() / total
france_share = region.loc[region.region.eq('EMEA - France'), 'spend_share'].iloc[0]
display(Markdown(
    f'### Commercial reading\n'
    f'- Top 10 account groups hold **{top10:.1%}** of additive recorded spend.\n'
    f'- France has **{france_share:.1%}** '    'of account-native recorded spend.\n'
    f'- **{metrics["parent_partner_conflict_groups"]:,}** parent groups mix known '    'partner ownership states; metadata needs review before assignment.\n'
    '- Source date sequencing and reported inactivity disagreement limit '    'any operational eligibility claim.'
))

## 7. Modeling implications
The real CSV has no account-period revenue, reactivation outcome or event time, timestamped ticket history, fiscal calendar, confirmed group consolidation rules, campaign costs, margins, or renewal assumptions. `sql/longitudinal_decay_template.sql` documents the missing real-data grain; it is intentionally not executed. The separate synthetic SQL demonstrates methodology only. `has_manual_review_exceptions=False` is quality triage, not outreach readiness; unknown required fields and fiscal eligibility need future fail-closed gates. Survival, Monte Carlo, and routing are outside Phase 1.